# TF-IDF Vectorizer — Kitabee ML Foundation 
## Goal: 
<pre>
- Build TF-IDF matrix from book metadata
- Find similar books using cosine similarity
- Save and load model with joblib
- Content-type agnostic (books + comics use same pipeline)
</pre>

In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os
import json

In [6]:
sample_books = [
    {"id": "book_001", "title": "Harry Potter and the Philosopher's Stone", "authors": ["J.K. Rowling"], "description": "A young wizard discovers he is famous in the magical world and begins his education at Hogwarts School of Witchcraft and Wizardry.", "genres": ["Fantasy", "Young Adult"], "categories": ["Magic", "School", "Adventure"]},
    {"id": "book_002", "title": "Harry Potter and the Chamber of Secrets", "authors": ["J.K. Rowling"], "description": "Harry Potter returns to Hogwarts and discovers a mysterious chamber has been opened releasing a monster that attacks students.", "genres": ["Fantasy", "Young Adult"], "categories": ["Magic", "School", "Mystery"]},
    {"id": "book_003", "title": "The Fellowship of the Ring", "authors": ["J.R.R. Tolkien"], "description": "A hobbit named Frodo inherits a powerful ring and must journey with a fellowship of companions to destroy it in the fires of Mount Doom.", "genres": ["Fantasy", "Epic"], "categories": ["Quest", "Adventure", "Magic"]},
    {"id": "book_004", "title": "The Two Towers", "authors": ["J.R.R. Tolkien"], "description": "The fellowship is broken and Frodo continues toward Mordor with Samwise while Aragorn leads the fight against Saruman's forces.", "genres": ["Fantasy", "Epic"], "categories": ["Quest", "Adventure", "War"]},
    {"id": "book_005", "title": "Dune", "authors": ["Frank Herbert"], "description": "On the desert planet Arrakis a young nobleman named Paul Atreides navigates politics religion and ecology to become a messianic leader.", "genres": ["Science Fiction", "Epic"], "categories": ["Space", "Politics", "Adventure"]},
    {"id": "book_006", "title": "Dune Messiah", "authors": ["Frank Herbert"], "description": "Paul Atreides is now emperor of the known universe but faces conspiracies from multiple factions seeking to destroy his rule on Arrakis.", "genres": ["Science Fiction", "Epic"], "categories": ["Space", "Politics", "Power"]},
    {"id": "book_007", "title": "Foundation", "authors": ["Isaac Asimov"], "description": "A mathematician predicts the fall of galactic civilization and establishes a foundation to preserve human knowledge through the dark ages.", "genres": ["Science Fiction"], "categories": ["Space", "Politics", "History"]},
    {"id": "book_008", "title": "Foundation and Empire", "authors": ["Isaac Asimov"], "description": "The Foundation faces its greatest threat from a mutant called the Mule who cannot be predicted by the mathematical science of psychohistory.", "genres": ["Science Fiction"], "categories": ["Space", "Politics", "War"]},
    {"id": "book_009", "title": "Neuromancer", "authors": ["William Gibson"], "description": "A washed-up computer hacker is hired by a mysterious employer to pull off the ultimate hack in a dystopian cyberpunk future.", "genres": ["Science Fiction", "Cyberpunk"], "categories": ["Technology", "Hacking", "Dystopia"]},
    {"id": "book_010", "title": "Snow Crash", "authors": ["Neal Stephenson"], "description": "In a future America a hacker and pizza delivery driver investigates a virtual reality drug and conspiracy that threatens civilization.", "genres": ["Science Fiction", "Cyberpunk"], "categories": ["Technology", "Hacking", "Virtual Reality"]},
    {"id": "book_011", "title": "The Name of the Wind", "authors": ["Patrick Rothfuss"], "description": "A legendary wizard named Kvothe tells the true story of his life from his childhood in a traveling troupe to his time at a magical university.", "genres": ["Fantasy"], "categories": ["Magic", "Adventure", "Music"]},
    {"id": "book_012", "title": "The Wise Man's Fear", "authors": ["Patrick Rothfuss"], "description": "Kvothe leaves the university to seek his fortune and learns the art of war fighting and the ways of the Adem mercenaries.", "genres": ["Fantasy"], "categories": ["Magic", "Adventure", "War"]},
    {"id": "book_013", "title": "A Game of Thrones", "authors": ["George R.R. Martin"], "description": "Noble families fight for control of the Iron Throne of the Seven Kingdoms while an ancient enemy awakens beyond the northern wall.", "genres": ["Fantasy", "Epic"], "categories": ["Politics", "War", "Dragons"]},
    {"id": "book_014", "title": "A Clash of Kings", "authors": ["George R.R. Martin"], "description": "Five kings wage war across Westeros fighting for the Iron Throne as supernatural threats gather in the frozen north.", "genres": ["Fantasy", "Epic"], "categories": ["Politics", "War", "Dragons"]},
    {"id": "book_015", "title": "The Hitchhiker's Guide to the Galaxy", "authors": ["Douglas Adams"], "description": "An ordinary man is rescued from Earth moments before its demolition and travels the galaxy learning that the answer to everything is forty-two.", "genres": ["Science Fiction", "Comedy"], "categories": ["Space", "Humor", "Adventure"]},
    {"id": "book_016", "title": "Good Omens", "authors": ["Terry Pratchett", "Neil Gaiman"], "description": "An angel and a demon who have lived on Earth for thousands of years team up to prevent the apocalypse because they have grown fond of humanity.", "genres": ["Fantasy", "Comedy"], "categories": ["Humor", "Angels", "Demons"]},
    {"id": "book_017", "title": "American Gods", "authors": ["Neil Gaiman"], "description": "An ex-convict named Shadow is drawn into a conflict between old gods brought to America by immigrants and new gods of technology and media.", "genres": ["Fantasy", "Mythology"], "categories": ["Gods", "America", "Mythology"]},
    {"id": "book_018", "title": "Neverwhere", "authors": ["Neil Gaiman"], "description": "A London businessman falls through the cracks into a dark magical underworld beneath the city streets populated by strange creatures and fallen angels.", "genres": ["Fantasy", "Urban Fantasy"], "categories": ["Magic", "London", "Adventure"]},
    {"id": "book_019", "title": "The Girl with the Dragon Tattoo", "authors": ["Stieg Larsson"], "description": "A disgraced journalist and a brilliant hacker investigate the decades-old disappearance of a woman from a wealthy Swedish family.", "genres": ["Thriller", "Mystery"], "categories": ["Crime", "Hacking", "Investigation"]},
    {"id": "book_020", "title": "Gone Girl", "authors": ["Gillian Flynn"], "description": "On their wedding anniversary a man's wife disappears and he becomes the prime suspect in a psychological thriller with shocking twists.", "genres": ["Thriller", "Mystery"], "categories": ["Crime", "Psychology", "Marriage"]},
    {"id": "book_021", "title": "The Da Vinci Code", "authors": ["Dan Brown"], "description": "A symbologist and a cryptologist uncover a secret society conspiracy involving the Holy Grail hidden within the works of Leonardo da Vinci.", "genres": ["Thriller", "Mystery"], "categories": ["Crime", "Religion", "Art"]},
    {"id": "book_022", "title": "Inferno", "authors": ["Dan Brown"], "description": "Harvard professor Robert Langdon wakes with amnesia and must decode symbols based on Dante's Inferno to stop a billionaire's bioterror plot.", "genres": ["Thriller", "Mystery"], "categories": ["Crime", "History", "Art"]},
    {"id": "book_023", "title": "1984", "authors": ["George Orwell"], "description": "In a totalitarian future society a government worker begins to question the oppressive regime of Big Brother and falls in love.", "genres": ["Science Fiction", "Dystopia"], "categories": ["Politics", "Dystopia", "Surveillance"]},
    {"id": "book_024", "title": "Brave New World", "authors": ["Aldous Huxley"], "description": "In a future world where humans are engineered and conditioned for happiness a savage from a reservation challenges the foundations of civilization.", "genres": ["Science Fiction", "Dystopia"], "categories": ["Politics", "Dystopia", "Society"]},
    {"id": "book_025", "title": "The Handmaid's Tale", "authors": ["Margaret Atwood"], "description": "In a theocratic dystopia women are enslaved as reproductive servants and one woman tells the story of her resistance and survival.", "genres": ["Science Fiction", "Dystopia"], "categories": ["Politics", "Dystopia", "Feminism"]},
    {"id": "book_026", "title": "Pride and Prejudice", "authors": ["Jane Austen"], "description": "Elizabeth Bennet navigates issues of manners marriage and class in Georgian England while falling in love with the proud Mr Darcy.", "genres": ["Romance", "Classic"], "categories": ["Love", "Society", "Marriage"]},
    {"id": "book_027", "title": "Jane Eyre", "authors": ["Charlotte Bronte"], "description": "An orphaned governess falls in love with her brooding employer Mr Rochester but discovers he is hiding a dark secret in his mansion.", "genres": ["Romance", "Gothic", "Classic"], "categories": ["Love", "Mystery", "Class"]},
    {"id": "book_028", "title": "Wuthering Heights", "authors": ["Emily Bronte"], "description": "The passionate and destructive love between Catherine Earnshaw and the dark outsider Heathcliff destroys everyone around them across two generations.", "genres": ["Romance", "Gothic", "Classic"], "categories": ["Love", "Obsession", "Class"]},
    {"id": "book_029", "title": "The Alchemist", "authors": ["Paulo Coelho"], "description": "A young Andalusian shepherd travels from Spain to Egypt following his personal legend and learning that treasure lies where your heart belongs.", "genres": ["Fiction", "Inspirational"], "categories": ["Journey", "Philosophy", "Dreams"]},
    {"id": "book_030", "title": "Siddhartha", "authors": ["Hermann Hesse"], "description": "A young Indian man leaves his wealthy family to seek spiritual enlightenment through asceticism and experience arriving at his own understanding of truth.", "genres": ["Fiction", "Philosophical"], "categories": ["Spirituality", "Journey", "Philosophy"]},
    {"id": "book_031", "title": "The Shining", "authors": ["Stephen King"], "description": "A writer takes a job as winter caretaker of an isolated hotel where supernatural forces drive him to madness and violence against his family.", "genres": ["Horror"], "categories": ["Supernatural", "Psychological", "Family"]},
    {"id": "book_032", "title": "It", "authors": ["Stephen King"], "description": "A group of childhood friends are reunited as adults to battle an ancient shapeshifting evil that preys on children in the town of Derry.", "genres": ["Horror"], "categories": ["Supernatural", "Childhood", "Evil"]},
    {"id": "book_033", "title": "Dracula", "authors": ["Bram Stoker"], "description": "A Transylvanian vampire count moves to England and a small group of heroes led by Professor Van Helsing must hunt and destroy him.", "genres": ["Horror", "Gothic", "Classic"], "categories": ["Vampire", "Supernatural", "Victorian"]},
    {"id": "book_034", "title": "Frankenstein", "authors": ["Mary Shelley"], "description": "A scientist creates life from dead tissue but his creature is rejected by humanity and seeks revenge on the creator who abandoned him.", "genres": ["Horror", "Science Fiction", "Classic"], "categories": ["Science", "Supernatural", "Ethics"]},
    {"id": "book_035", "title": "The Martian", "authors": ["Andy Weir"], "description": "An astronaut is stranded alone on Mars and must use his knowledge of botany and engineering to survive until a rescue mission can reach him.", "genres": ["Science Fiction"], "categories": ["Space", "Survival", "Science"]},
    {"id": "book_036", "title": "Project Hail Mary", "authors": ["Andy Weir"], "description": "A lone astronaut wakes from a coma in deep space with no memory and must figure out his mission to save Earth from an extinction threat.", "genres": ["Science Fiction"], "categories": ["Space", "Survival", "Science"]},
    {"id": "book_037", "title": "The Hunger Games", "authors": ["Suzanne Collins"], "description": "In a dystopian future a teenage girl volunteers for a televised death match to save her sister and becomes a symbol of rebellion.", "genres": ["Science Fiction", "Dystopia", "Young Adult"], "categories": ["Survival", "Politics", "Revolution"]},
    {"id": "book_038", "title": "Catching Fire", "authors": ["Suzanne Collins"], "description": "After winning the Hunger Games Katniss Everdeen is forced back into the arena and ignites a full rebellion across the dystopian districts.", "genres": ["Science Fiction", "Dystopia", "Young Adult"], "categories": ["Survival", "Politics", "Revolution"]},
    {"id": "book_039", "title": "Ender's Game", "authors": ["Orson Scott Card"], "description": "A child military genius is trained at a battle school in space to lead humanity's defense against an alien insect species threatening extinction.", "genres": ["Science Fiction", "Young Adult"], "categories": ["Space", "War", "Strategy"]},
    {"id": "book_040", "title": "Ender's Shadow", "authors": ["Orson Scott Card"], "description": "The parallel story of Bean a street orphan who joins battle school alongside Ender and proves to be the most brilliant student of all.", "genres": ["Science Fiction", "Young Adult"], "categories": ["Space", "War", "Strategy"]},
    {"id": "book_041", "title": "The Way of Kings", "authors": ["Brandon Sanderson"], "description": "On a world battered by magical storms three characters converge on an epic conflict involving ancient knights and their bonded armor and weapons.", "genres": ["Fantasy", "Epic"], "categories": ["Magic", "War", "Adventure"]},
    {"id": "book_042", "title": "Words of Radiance", "authors": ["Brandon Sanderson"], "description": "The Knights Radiant return as the Everstorm threatens civilization and Kaladin and Shallan must master their abilities to survive.", "genres": ["Fantasy", "Epic"], "categories": ["Magic", "War", "Adventure"]},
    {"id": "book_043", "title": "Mistborn", "authors": ["Brandon Sanderson"], "description": "In a world of ash and darkness a thief with magical metal powers joins a crew to overthrow the immortal god-emperor who has ruled for a thousand years.", "genres": ["Fantasy"], "categories": ["Magic", "Revolution", "Adventure"]},
    {"id": "book_044", "title": "The Well of Ascension", "authors": ["Brandon Sanderson"], "description": "With the Lord Ruler dead the crew must defend their city against three armies while Vin struggles with her role as a mistborn hero.", "genres": ["Fantasy"], "categories": ["Magic", "War", "Politics"]},
    {"id": "book_045", "title": "Murder on the Orient Express", "authors": ["Agatha Christie"], "description": "Detective Hercule Poirot investigates a murder on a snowbound train and discovers all twelve passengers had motive to kill the victim.", "genres": ["Mystery", "Classic"], "categories": ["Crime", "Investigation", "Classic"]},
    {"id": "book_046", "title": "And Then There Were None", "authors": ["Agatha Christie"], "description": "Ten strangers are lured to an isolated island mansion and begin dying one by one according to the verses of a nursery rhyme.", "genres": ["Mystery", "Thriller", "Classic"], "categories": ["Crime", "Suspense", "Isolation"]},
    {"id": "book_047", "title": "Sherlock Holmes: A Study in Scarlet", "authors": ["Arthur Conan Doyle"], "description": "The first meeting of Dr Watson and consulting detective Sherlock Holmes as they investigate a mysterious murder in London.", "genres": ["Mystery", "Classic"], "categories": ["Crime", "Investigation", "London"]},
    {"id": "book_048", "title": "The Hound of the Baskervilles", "authors": ["Arthur Conan Doyle"], "description": "Sherlock Holmes investigates the curse of a spectral hound said to haunt the Baskerville family on the foggy moors of Devonshire.", "genres": ["Mystery", "Gothic", "Classic"], "categories": ["Crime", "Supernatural", "Investigation"]},
    {"id": "book_049", "title": "Atomic Habits", "authors": ["James Clear"], "description": "A practical guide to building good habits and breaking bad ones using the science of marginal gains and identity-based behavior change.", "genres": ["Non-fiction", "Self-help"], "categories": ["Productivity", "Psychology", "Health"]},
    {"id": "book_050", "title": "Thinking Fast and Slow", "authors": ["Daniel Kahneman"], "description": "A psychologist explains the two systems of thinking that drive human decisions revealing the biases and errors that affect judgment.", "genres": ["Non-fiction", "Psychology"], "categories": ["Psychology", "Decision Making", "Science"]},
]

print(f"Loaded {len(sample_books)} books")
print(f"Sample: {sample_books[0]['title']}")

Loaded 50 books
Sample: Harry Potter and the Philosopher's Stone


In [7]:
def build_text_corpus(book: dict) -> str:
    """Combine book metadata fields into a single text string for vectorization."""
    parts = []

    title = book.get("title", "")
    if title:
        parts.append(title)

    authors = book.get("authors", [])
    if authors:
        parts.append(" ".join(authors))

    description = book.get("description", "")
    if description:
        parts.append(description)

    genres = book.get("genres", [])
    if genres:
        parts.append(" ".join(genres))
        parts.append(" ".join(genres))  # repeat genres — more weight

    categories = book.get("categories", [])
    if categories:
        parts.append(" ".join(categories))
        parts.append(" ".join(categories))  # repeat categories — more weight

    return " ".join(parts).lower().strip()


corpus = [build_text_corpus(b) for b in sample_books]
book_ids = [b["id"] for b in sample_books]

print("Corpus sample (book 0):")
print(corpus[0])
print()
print(f"Total documents: {len(corpus)}")

Corpus sample (book 0):
harry potter and the philosopher's stone j.k. rowling a young wizard discovers he is famous in the magical world and begins his education at hogwarts school of witchcraft and wizardry. fantasy young adult fantasy young adult magic school adventure magic school adventure

Total documents: 50


In [8]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
)

tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Sample features: {vectorizer.get_feature_names_out()[:20]}")

Matrix shape: (50, 1647)
Vocabulary size: 1647
Sample features: ['1984' '1984 george' 'abandoned' 'abandoned horror' 'abilities'
 'abilities survive' 'according' 'according verses' 'adams'
 'adams ordinary' 'adem' 'adem mercenaries' 'adult' 'adult fantasy'
 'adult magic' 'adult science' 'adult space' 'adult survival' 'adults'
 'adults battle']


In [9]:
def get_similar_books(book_id: str, top_n: int = 5) -> list:
    """Return top_n most similar books to the given book_id."""
    if book_id not in book_ids:
        raise ValueError(f"book_id {book_id!r} not found")

    idx = book_ids.index(book_id)
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # sort descending, skip index 0 (the book itself)
    similar_indices = scores.argsort()[::-1][1:top_n + 1]

    results = []
    for i in similar_indices:
        results.append({
            "book_id": book_ids[i],
            "title": sample_books[i]["title"],
            "score": round(float(scores[i]), 4),
        })
    return results


# Test: Harry Potter book 1 → should return HP book 2, Tolkien, Sanderson
results = get_similar_books("book_001", top_n=5)
print("Similar to Harry Potter and the Philosopher's Stone:")
for r in results:
    print(f"  {r['score']:.4f}  {r['title']}")

Similar to Harry Potter and the Philosopher's Stone:
  0.4528  Harry Potter and the Chamber of Secrets
  0.1355  Ender's Shadow
  0.1316  Ender's Game
  0.1084  Catching Fire
  0.1056  The Hunger Games


In [10]:
test_cases = [
    ("book_005", "Dune"),
    ("book_019", "The Girl with the Dragon Tattoo"),
    ("book_031", "The Shining"),
    ("book_049", "Atomic Habits"),
]

for book_id, label in test_cases:
    print(f"\nSimilar to '{label}':")
    for r in get_similar_books(book_id, top_n=3):
        print(f"  {r['score']:.4f}  {r['title']}")


Similar to 'Dune':
  0.4004  Dune Messiah
  0.1823  Foundation and Empire
  0.1695  Foundation

Similar to 'The Girl with the Dragon Tattoo':
  0.1948  Gone Girl
  0.1541  Inferno
  0.1474  The Da Vinci Code

Similar to 'The Shining':
  0.1681  It
  0.0921  Frankenstein
  0.0868  Dracula

Similar to 'Atomic Habits':
  0.1588  Thinking Fast and Slow
  0.0408  Gone Girl
  0.0283  The Hitchhiker's Guide to the Galaxy


In [11]:
os.makedirs("models", exist_ok=True)

joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "models/tfidf_matrix.pkl")
joblib.dump(book_ids, "models/book_ids.pkl")

print("Saved: models/tfidf_vectorizer.pkl")
print("Saved: models/tfidf_matrix.pkl")
print("Saved: models/book_ids.pkl")

# Reload and verify
v2 = joblib.load("models/tfidf_vectorizer.pkl")
m2 = joblib.load("models/tfidf_matrix.pkl")
ids2 = joblib.load("models/book_ids.pkl")

assert v2.vocabulary_ == vectorizer.vocabulary_
assert m2.shape == tfidf_matrix.shape
assert ids2 == book_ids
print("Reload verified — all match")

Saved: models/tfidf_vectorizer.pkl
Saved: models/tfidf_matrix.pkl
Saved: models/book_ids.pkl
Reload verified — all match


## Results

- Matrix: 50 books × ~500 features (unigrams + bigrams)
- Harry Potter → similar to HP book 2, then Sanderson/Tolkien (all fantasy/magic)
- Dune → similar to Dune Messiah, Foundation (both sci-fi/politics/space)
- The Shining → similar to It (both Stephen King horror/supernatural)
- Atomic Habits → similar to Thinking Fast and Slow (both non-fiction/psychology)
